**CI twin of `ch10-mlp-from-scratch.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
import math

class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = _prev
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def __pow__(self, k):
        out = Value(self.data ** k, (self,), f"**{k}")
        def _backward():
            self.grad += out.grad * k * self.data ** (k - 1)
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")
        def _backward():
            self.grad += out.grad * out.data
        out._backward = _backward
        return out

    def log(self):
        out = Value(math.log(self.data), (self,), "log")
        def _backward():
            self.grad += out.grad / self.data      # d(log x)/dx = 1/x
        out._backward = _backward
        return out

    def relu(self):
        out = Value(self.data if self.data > 0 else 0.0, (self,), "relu")
        def _backward():
            self.grad += out.grad * (1.0 if self.data > 0 else 0.0)
        out._backward = _backward
        return out

    def __neg__(self):           return self * -1
    def __sub__(self, other):    return self + (-other if isinstance(other, Value) else -other)
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1
    def __radd__(self, other):   return self + other
    def __rmul__(self, other):   return self * other

    def backward(self):
        order, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for parent in v._prev:
                    build(parent)
                order.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

print("engine ready: + * ** exp log relu, backward()")

In [ ]:
import random

class Neuron:
    def __init__(self, n_in, relu=True):
        k = n_in ** -0.5                 # small random start, scaled by fan-in
        self.w = [Value(random.uniform(-k, k)) for _ in range(n_in)]
        self.b = Value(0.0)
        self.use_relu = relu

    def __call__(self, x):
        z = self.b
        for wi, xi in zip(self.w, x):
            z = z + wi * xi
        return z.relu() if self.use_relu else z

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, n_in, n_out, relu=True):
        self.neurons = [Neuron(n_in, relu) for _ in range(n_out)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, sizes):
        self.layers = [Layer(sizes[i], sizes[i + 1],
                             relu=(i < len(sizes) - 2))
                       for i in range(len(sizes) - 1)]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

random.seed(0)
net = MLP([64, 16, 10])
print("knobs:", len(net.parameters()))

In [ ]:
def cross_entropy(logits, target):
    zmax = max(z.data for z in logits)          # a constant shift — no grad path
    exps = [(z - zmax).exp() for z in logits]
    total = exps[0]
    for e in exps[1:]:
        total = total + e
    return -(exps[target] / total).log()

probe = cross_entropy(net([0.5] * 64), 3)
order, seen = [], set()
def count(v):
    if v not in seen:
        seen.add(v)
        for p in v._prev:
            count(p)
        order.append(v)
count(probe)
print(f"one sample's loss builds a graph of {len(order)} nodes")

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(
    digits.data, digits.target, test_size=0.25,
    random_state=42, stratify=digits.target)

N = 150
Xs = [[v / 16.0 for v in row] for row in Xtr[:N]]
ys = [int(t) for t in ytr[:N]]
Xq = [[v / 16.0 for v in row] for row in Xte[:100]]
yq = [int(t) for t in yte[:100]]

first = sum(cross_entropy(net(x), t).data
            for x, t in zip(Xs[:25], ys[:25])) / 25
print(f"untrained loss on the first batch: {first:.4f}")
print(f"ln(10)                           : {math.log(10):.4f}")

In [ ]:
import time

params = net.parameters()
velocity = [0.0] * len(params)
curve = []

t0 = time.time()
for epoch in range(5):
    epoch_loss = 0.0
    for s in range(0, N, 25):
        xb, yb = Xs[s:s + 25], ys[s:s + 25]

        for p in params:                 # 1. zero_grad  (Ch9)
            p.grad = 0.0
        loss = cross_entropy(net(xb[0]), yb[0])
        for x, t in zip(xb[1:], yb[1:]): # 2–3. forward + loss (Ch4, Ch5)
            loss = loss + cross_entropy(net(x), t)
        loss = loss * (1.0 / 25)
        loss.backward()                  # 4. backward   (Ch6, Ch9)
        for i, p in enumerate(params):   # 5. momentum step (Ch7)
            velocity[i] = 0.9 * velocity[i] + p.grad
            p.data -= 0.2 * velocity[i]

        epoch_loss += loss.data
    curve.append(epoch_loss / (N // 25))
    print(f"epoch {epoch + 1}: mean loss {curve[-1]:.3f}")

print(f"trained in {time.time() - t0:.0f} s")

In [ ]:
hits = 0
preds = []
for xi, yi in zip(Xq, yq):
    logits = net(xi)
    guess = max(range(10), key=lambda i: logits[i].data)
    preds.append(guess)
    hits += (guess == yi)

print(f"test accuracy: {hits}/{len(yq)}")
print("first ten predictions:", preds[:10])
print("first ten truths     :", yq[:10])
miss = next(i for i in range(len(yq)) if preds[i] != yq[i])
print(f"first mistake: read a {yq[miss]} as a {preds[miss]}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3.2),
                               gridspec_kw={"width_ratios": [2, 1]})
ax1.plot(range(1, 6), curve, marker="o", color="#2563eb")
ax1.axhline(math.log(10), linestyle="--", color="#94a3b8")
ax1.text(1.05, math.log(10) + 0.05, "ln(10) — uniform ignorance",
         fontsize=8, color="#64748b")
ax1.set_xlabel("epoch"); ax1.set_ylabel("mean cross-entropy")
ax1.set_title("Your engine, learning")

ax2.imshow(np.array(Xte[miss]).reshape(8, 8), cmap="gray_r")
ax2.set_title(f"read a {yq[miss]} as a {preds[miss]}", fontsize=9)
ax2.set_xticks([]); ax2.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
t0 = time.time()
for x, t in zip(Xs[:25], ys[:25]):
    for p in params:
        p.grad = 0.0
    cross_entropy(net(x), t).backward()
per_sample = (time.time() - t0) / 25

node_scale = (784 * 128 + 128 * 10) / (64 * 16 + 16 * 10)
mnist_hours = per_sample * node_scale * 60000 / 3600
print(f"per sample here : {per_sample * 1000:.0f} ms "
      f"(a {len(order)}-node graph, scalar Python)")
print(f"one MNIST epoch : ~{node_scale:.0f}× the nodes × 60,000 samples "
      f"≈ {mnist_hours:.0f} hours")

In [ ]:
w = [Value(0.5), Value(-0.25)]
b = Value(0.1)

def neuron_call(x):
    z = b
    for wi, xi in zip(w, x):
        z = z + wi * xi
    return z.relu()

live = neuron_call([1.0, 2.0])
dead = neuron_call([-1.0, 0.0])

run_tests([
    ("gate open", round(live.data, 4), 0.1),
    ("gate shut", round(dead.data, 4), 0.0),
])

In [ ]:
from lib.grader import decreased, changed, between   # the property builders

def zero_grad(params):
    for p in params:
        p.grad = 0.0

def sgd_step(params, lr):
    for p in params:
        p.data -= lr * p.grad

w1, w2, b = Value(0.5), Value(-0.5), Value(0.0)
params = [w1, w2, b]
pts = [((1.0, 2.0), 3.0), ((2.0, 0.0), 2.0)]

losses = []
for step in range(15):
    zero_grad(params)
    loss = Value(0.0)
    for (xa, xb), t in pts:
        pred = w1 * xa + w2 * xb + b
        loss = loss + (pred - t) ** 2
    loss = loss * 0.5
    losses.append(loss.data)
    loss.backward()
    sgd_step(params, 0.1)

run_tests([
    decreased("training loss fell ≥90%", losses[0], losses[-1], min_drop=0.9),
    changed("all three knobs moved", [0.5, -0.5, 0.0],
            [p.data for p in params]),
    between("final loss near zero", losses[-1], 0.0, 0.01),
])